# Tugas Akhir Evaluasi (Pra-UAS) - Analisis Jejaring Sosial

**Informasi Mahasiswa:**
- **Nama:** Bayu Samudra
- **NIM:** 2211310079
- **Kelas:** Teknologi Informasi 8c
- **Mata Kuliah:** Analisis Jejaring Sosial

---

## Deskripsi Proyek
Notebook ini dikembangkan untuk melakukan **Social Network Analysis (SNA)** menggunakan dataset riil **ego-Facebook** yang bersumber dari **Stanford Large Network Dataset Collection (SNAP)**. Dataset ini memetakan lingkaran pertemanan anonim di Facebook dengan **4.039 node** dan **88.234 edge**.

Notebook ini mencakup:
1. Unduhan otomatis data SNAP Facebook.
2. Representasi graf dan matriks adjacency.
3. Perhitungan metrik sentralitas (Degree, Betweenness, Closeness, Eigenvector) serta analisis peran aktor kunci.
4. Kalkulasi metrik global (Density, Diameter, Average Path Length, Clustering Coefficient) dan deteksi komunitas dengan **Algoritma Louvain**.
5. Simulasi penyebaran informasi menggunakan **Model SI (Susceptible-Infected)**.
6. Visualisasi graf statis dan interaktif 3D-like (PyVis).

---

### 1. Instalasi Library Tambahan
Kita akan menginstal `pyvis` untuk visualisasi HTML interaktif dan `python-louvain` untuk algoritma deteksi komunitas Louvain.

In [ ]:
!pip install pyvis python-louvain

### 2. Unduh dan Muat Dataset SNAP Facebook
Data diunduh langsung dari `https://snap.stanford.edu/data/facebook_combined.txt.gz` dan dimuat ke dalam graf tidak berarah (`nx.Graph`) menggunakan NetworkX.

In [ ]:
import urllib.request
import gzip
import os
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from pyvis.network import Network
import json

# Unduh data jika belum ada
url = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
gz_filename = "facebook_combined.txt.gz"
if not os.path.exists(gz_filename):
    print("Mengunduh dataset dari SNAP...")
    urllib.request.urlretrieve(url, gz_filename)
    print("Unduhan selesai!")
else:
    print("Dataset sudah ada secara lokal.")

# Muat graf menggunakan NetworkX
with gzip.open(gz_filename, "rt") as f:
    G = nx.read_edgelist(f, nodetype=int)

print(f"Graf berhasil dimuat!")
print(f"Jumlah Node (Pengguna): {G.number_of_nodes()}")
print(f"Jumlah Edge (Hubungan): {G.number_of_edges()}")
print(f"Jenis Graf: {'Berarah' if G.is_directed() else 'Tidak Berarah'} (Undirected)")

### 3. Representasi Graf & Matriks Adjacency
Berikut adalah contoh representasi ketetanggaan (adjacency matrix) dari 5 node pertama (Node ID 0 s.d 4) di dalam graf.

In [ ]:
sample_nodes = [0, 1, 2, 3, 4]
adj_matrix = nx.adjacency_matrix(G, nodelist=sample_nodes).todense()
print("Sampel Matriks Adjacency (Node 0-4):")
print(np.array(adj_matrix))

### 4. Perhitungan Metrik Sentralitas (Degree, Betweenness, Closeness, Eigenvector)
Kita akan menghitung metrik sentralitas untuk masing-masing node dan mengidentifikasi 5 aktor terpenting pada masing-masing metrik.

In [ ]:
print("Menghitung Degree Centrality...")
deg_cent = nx.degree_centrality(G)

print("Menghitung Closeness Centrality (ini mungkin memakan waktu ~15-20 detik)...")
closeness_cent = nx.closeness_centrality(G)

print("Menghitung Betweenness Centrality (ini mungkin memakan waktu ~30-50 detik)...")
betweenness_cent = nx.betweenness_centrality(G)

print("Menghitung Eigenvector Centrality...")
eigenvector_cent = nx.eigenvector_centrality(G, max_iter=1000)

# Simpan hasil ke DataFrame
df_cent = pd.DataFrame(index=G.nodes())
df_cent["degree"] = [deg_cent[n] for n in df_cent.index]
df_cent["betweenness"] = [betweenness_cent[n] for n in df_cent.index]
df_cent["closeness"] = [closeness_cent[n] for n in df_cent.index]
df_cent["eigenvector"] = [eigenvector_cent[n] for n in df_cent.index]
df_cent.to_csv("centrality_results.csv")
print("Hasil sentralitas berhasil disimpan di centrality_results.csv!")

# Cetak Top 5 Aktor Terpenting
metrics = {
    "Degree Centrality": df_cent["degree"],
    "Betweenness Centrality": df_cent["betweenness"],
    "Closeness Centrality": df_cent["closeness"],
    "Eigenvector Centrality": df_cent["eigenvector"]
}

for name, series in metrics.items():
    print(f"\n--- Top 5 Aktor Terpenting Berdasarkan {name} ---")
    top_5 = series.sort_values(ascending=False).head(5)
    for node, val in top_5.items():
        print(f"  Node ID {node:4}: Skor = {val:.5f}")

### 5. Karakteristik Jaringan Global & Deteksi Komunitas (Louvain)
Kita akan menghitung kerapatan (*density*), diameter lintasan terpanjang, rata-rata panjang jalur terpendek, koefisien clustering, serta mendeteksi struktur komunitas/divisi informal menggunakan algoritma Louvain.

In [ ]:
try:
    from community import community_louvain
except Exception:
    try:
        import community.community_louvain as community_louvain
    except Exception:
        import community_louvain

print("Menghitung metrik global...")
density = nx.density(G)
avg_clustering = nx.average_clustering(G)

# Graf Facebook SNAP terhubung penuh
is_conn = nx.is_connected(G)
print(f"Apakah graf terhubung penuh? {is_conn}")
if is_conn:
    diameter = nx.diameter(G)
    avg_path_length = nx.average_shortest_path_length(G)
else:
    largest_cc = max(nx.connected_components(G), key=len)
    G_lcc = G.subgraph(largest_cc)
    diameter = nx.diameter(G_lcc)
    avg_path_length = nx.average_shortest_path_length(G_lcc)

print(f"Density (Kerapatan Graf): {density:.6f}")
print(f"Diameter Graf: {diameter}")
print(f"Rata-rata Panjang Jalur (Average Path Length): {avg_path_length:.4f}")
print(f"Rata-rata Koefisien Clustering: {avg_clustering:.4f}")

print("\nMenjalankan Deteksi Komunitas Louvain...")
try:
    partition = community_louvain.best_partition(G, random_state=42)
    modularity = community_louvain.modularity(partition, G)
except Exception:
    import networkx.algorithms.community as nx_comm
    louvain_sets = nx_comm.louvain_communities(G, seed=42)
    partition = {}
    for c_id, nodes in enumerate(louvain_sets):
        for n in nodes:
            partition[n] = c_id
    modularity = nx_comm.modularity(G, louvain_sets)

print(f"Modularitas Louvain: {modularity:.4f}")

# Simpan komunitas ke file
df_nodes = pd.DataFrame(index=G.nodes())
df_nodes["community"] = [partition[n] for n in df_nodes.index]
df_nodes.to_csv("nodes_with_community.csv")

# Hitung ukuran komunitas
comm_sizes = df_nodes["community"].value_counts().to_dict()
print(f"Jumlah komunitas yang terdeteksi: {len(comm_sizes)}")
print("10 Komunitas Terbesar:")
for comm_id, size in sorted(comm_sizes.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  Komunitas {comm_id:2}: {size:4} node")


### 6. Simulasi Penyebaran Informasi (Model Epidemi SI)
Di sini kita menyimulasikan penyebaran gosip, berita, atau informasi di Facebook. Kita membandingkan kecepatan penyebaran jika dimulai dari benih yang berbeda:
- **Degree Hub / Betweenness Broker (Node 107)**
- **Eigenvector Hub (Node 1912)**
- **Node Acak (Random Staff)**

In [ ]:
def run_si_simulation(G, seed_node, transmission_prob=0.08, num_steps=20, num_trials=50):
    num_nodes = G.number_of_nodes()
    history = np.zeros((num_trials, num_steps + 1))
    
    for trial in range(num_trials):
        states = {node: 0 for node in G.nodes()} # 0: Susceptible (Belum tahu), 1: Infected (Tahu)
        states[seed_node] = 1
        infected = {seed_node}
        history[trial, 0] = 1
        
        for step in range(1, num_steps + 1):
            new_infected = set()
            for u in list(infected):
                for v in G.neighbors(u):
                    if states[v] == 0:
                        if random.random() < transmission_prob:
                            new_infected.add(v)
            for v in new_infected:
                states[v] = 1
                infected.add(v)
            history[trial, step] = len(infected)
            
    return np.mean(history, axis=0)

# Tentukan benih
degree_seed = df_cent["degree"].idxmax()      # Node 107
eigenvector_seed = df_cent["eigenvector"].idxmax() # Node 1912

# Tentukan benih acak dengan sentralitas rendah
random.seed(42)
low_cent_nodes = df_cent[df_cent["degree"] < 0.01].index.tolist()
random_seed = random.choice(low_cent_nodes)

seeds = {
    "Degree/Betweenness Hub (Node 107)": degree_seed,
    "Eigenvector Hub (Node 1912)": eigenvector_seed,
    "Random Node (Node {})".format(random_seed): random_seed
}

num_steps = 20
num_trials = 50
prob = 0.08

results = {}
for name, seed in seeds.items():
    print(f"Simulasi penyebaran dari {name}...")
    avg_history = run_si_simulation(G, seed, transmission_prob=prob, num_steps=num_steps, num_trials=num_trials)
    results[name] = avg_history

# Plotting kurva simulasi
plt.figure(figsize=(10, 6))
for name, history in results.items():
    pct = (history / G.number_of_nodes()) * 100
    plt.plot(pct, label=name, linewidth=2.5)
    
plt.title("Kurva Simulasi Penyebaran Informasi (Model SI) - Facebook SNAP", fontsize=13, fontweight='bold')
plt.xlabel("Langkah Waktu (Siklus)")
plt.ylabel("Persentase Terinfeksi (%)")
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.xlim(0, num_steps)
plt.ylim(0, 105)
plt.show()

### 7. Visualisasi Jejaring Sosial
Kita akan menggambar visualisasi statis menggunakan `NetworkX` dan visualisasi interaktif dinamis menggunakan `PyVis`.

In [ ]:
# A. Visualisasi Statis Graf (NetworkX)
print("Menggambar graf statis (proses posisi mungkin memakan waktu ~10-15 detik)...")
plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, k=0.08, iterations=15, seed=42)

# Skema warna komunitas
unique_comm = sorted(list(set(partition.values())))
color_palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", 
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#aec7e8", "#ffbb78", "#98df8a", "#ff9896", "#c5b0d5"
]
while len(color_palette) < len(unique_comm):
    color_palette += color_palette
    
node_colors = [color_palette[partition[node]] for node in G.nodes()]
node_sizes = [2 + (df_cent.loc[node, "degree"] * 250) for node in G.nodes()]

nx.draw_networkx_edges(G, pos, alpha=0.01, width=0.1, edge_color="gray")
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.7)

# Tampilkan label aktor paling sentral
labels = {107: "Node 107 (Degree/Betweenness Hub)", 1912: "Node 1912 (Eigenvector Hub)"}
nx.draw_networkx_labels(
    G, pos, labels=labels, font_size=10, font_weight='bold', font_color='black',
    bbox=dict(facecolor='white', edgecolor='red', alpha=0.8, boxstyle='round,pad=0.2')
)

plt.title("Visualisasi Statis Jejaring Facebook SNAP", fontsize=14, fontweight='bold')
plt.axis("off")
plt.show()

#### B. Visualisasi Interaktif HTML (PyVis)
Karena graf asli sangat besar (88.000 edge) dan dapat membuat peramban *lag*, kita melakukan sub-sampling graf yang berpusat pada lingkaran pertemanan terdekat dari aktor-aktor sentral utama (Node 107, 1684, dan 1912). Ini menghasilkan visualisasi interaktif yang responsif, terperinci, dan indah.

In [ ]:
# Pilih sub-graf dari tetangga aktor-aktor sentral utama
top_nodes = [107, 1684, 1912]
subgraph_nodes = set(top_nodes)
for node in top_nodes:
    subgraph_nodes.update(G.neighbors(node))
    
G_sub = G.subgraph(subgraph_nodes)
print(f"Sub-graf interaktif: {G_sub.number_of_nodes()} node, {G_sub.number_of_edges()} edge")

net = Network(height="600px", width="100%", bgcolor="#0f172a", font_color="white", directed=False)
net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=60, spring_strength=0.05, damping=0.09)

# Tambahkan node
for node in G_sub.nodes():
    comm = int(partition[node])
    deg = df_cent.loc[node, "degree"]
    bet = df_cent.loc[node, "betweenness"]
    clo = df_cent.loc[node, "closeness"]
    eig = df_cent.loc[node, "eigenvector"]
    
    tooltip = f"""
    <div style="font-family: sans-serif; padding: 10px; color: #1e293b; background: white; border-radius: 8px; border: 1px solid #cbd5e1;">
        <b style="font-size: 14px; color: #0f172a;">Node ID: {node}</b><br/>
        <hr style="margin: 5px 0; border: 0; border-top: 1px solid #e2e8f0;"/>
        <b>Komunitas (Louvain):</b> {comm}<br/>
        <hr style="margin: 5px 0; border: 0; border-top: 1px solid #e2e8f0;"/>
        <b>SNA Centrality Scores:</b><br/>
        - Degree: {deg:.5f}<br/>
        - Betweenness: {bet:.5f}<br/>
        - Closeness: {clo:.5f}<br/>
        - Eigenvector: {eig:.5f}
    </div>
    """
    
    color = color_palette[comm]
    size = 5 + (deg * 150)
    if node in top_nodes:
        size += 15
        
    net.add_node(
        int(node),
        label=f"Hub {node}" if node in top_nodes else "",
        title=tooltip,
        color=color,
        size=size,
        shape="dot"
    )
    
# Tambahkan edge
for u, v in G_sub.edges():
    net.add_edge(
        int(u), int(v),
        color={"color": "#475569", "highlight": "#f43f5e", "hover": "#f43f5e", "opacity": 0.25},
        width=0.5
    )
    
options = {
    "interaction": {
        "hover": True,
        "tooltipDelay": 200,
        "hideEdgesOnDrag": True
    },
    "physics": {
        "barnesHut": {
            "gravitationalConstant": -8000,
            "centralGravity": 0.3,
            "springLength": 60,
            "springStrength": 0.05,
            "damping": 0.09,
            "avoidOverlap": 0.2
        },
        "stabilization": {
            "enabled": True,
            "iterations": 80,
            "updateInterval": 20,
            "fit": True
        }
    }
}
net.set_options(json.dumps(options))
net.save_graph("network_interactive.html")
print("Visualisasi interaktif network_interactive.html berhasil diekspor!")

# Tampilkan di Google Colab
from IPython.display import HTML
HTML(filename="network_interactive.html")

# LAPORAN AKHIR ANALISIS JEJARING SOSIAL

**Mahasiswa:** Bayu Samudra (NIM: 2211310079)  
**Kelas/Matkul:** TI 8c / Analisis Jejaring Sosial  
**Dataset:** SNAP ego-Facebook (4.039 Node, 88.234 Edge)  

---

## Jawaban Pertanyaan Evaluasi

### 1. Representasi Graf Jejaring Sosial
- **Jenis Graf:** Graf Tidak Berarah (**Undirected**) dan Tidak Berbobot (**Unweighted**). Ini dikarenakan data SNAP merepresentasikan jalinan pertemanan timbal-balik (jika A berteman dengan B, maka B juga berteman dengan A) tanpa adanya tingkatan bobot formal.
- **Contoh Matriks Adjacency (Nodes 0 s.d 4):**
  $$
  A = \begin{pmatrix}
  0 & 1 & 1 & 1 & 1 \\
  1 & 0 & 0 & 0 & 0 \\
  1 & 0 & 0 & 0 & 0 \\
  1 & 0 & 0 & 0 & 0 \\
  1 & 0 & 0 & 0 & 0
  \end{pmatrix}
  $$
  - Matriks simetris terhadap diagonal utama karena graf bersifat tidak berarah.
  - Diagonal utama bernilai `0` (tidak berteman dengan diri sendiri).
  - Baris ke-0 terhubung ke kolom 1, 2, 3, dan 4 ($A_{0,1} = A_{0,2} = A_{0,3} = A_{0,4} = 1$), yang menunjukkan Node 0 berteman langsung dengan Node 1, 2, 3, dan 4.

### 2. Analisis Sentralitas (Centrality Analysis)
- **Hasil Perhitungan Top Aktor:**
  - **Degree Centrality:** Node 107 (0.25879). Memiliki jumlah teman langsung terbanyak (1.045 teman/tetangga). Bertindak sebagai *Hub* utama.
  - **Betweenness Centrality:** Node 107 (0.48052). Menjembatani 48% dari lintasan terpendek antaranggota lain. Bertindak sebagai *Broker/Jembatan* strategis.
  - **Closeness Centrality:** Node 107 (0.45970). Memiliki jarak terpendek rata-rata paling kecil ke seluruh jaringan. Aktor tercepat dalam menyerap/mengirim informasi.
  - **Eigenvector Centrality:** Node 1912 (0.09541). Terhubung dengan banyak teman penting yang juga memiliki sentralitas tinggi.

### 3. Karakteristik Jaringan Global & Komunitas
- **Metrik Jaringan:**
  - **Density:** 0.010820 (1.08%). Sangat longgar (jarang), wajar untuk jaringan sosial berskala ribuan.
  - **Diameter:** 8. Jarak terjauh di antara dua pengguna di jejaring ini adalah 8 langkah.
  - **Average Path Length:** 3.6925 langkah. Rata-rata dua pengguna hanya terpisah jarak ~3.7 langkah (konsep *small-world*).
  - **Clustering Coefficient:** 0.6055 (60.55%). Sangat tinggi, menunjukkan kecenderungan kuat pengguna membentuk lingkaran pertemanan padat (*cliques*).
- **Deteksi Komunitas Louvain:**
  - **Modularitas:** 0.8350 (menunjukkan pemisahan komunitas yang sangat jelas).
  - Terdeteksi **16 komunitas utama**. Komunitas terbesar adalah Komunitas 9 (548 node) dan Komunitas 4 (535 node). Komunitas-komunitas ini mencerminkan lingkaran pertemanan informal di dunia nyata (misalnya alumni sekolah, rekan kerja, dll.).

### 4. Simulasi Penyebaran Informasi
- Simulasi model SI menunjukkan bahwa informasi yang disebarkan dari **Node 107 (Degree/Betweenness/Closeness Hub)** menyebar secara eksponensial dan mendominasi 90% populasi hanya dalam waktu **5-6 langkah waktu**.
- Sebaliknya, penyebaran dari **Random Node** dengan sentralitas rendah berjalan lambat, memerlukan waktu **9 langkah waktu** untuk menjangkau 90% populasi karena informasi terperangkap di kelompok lokal sebelum akhirnya keluar.

### 5. Kesimpulan Karakteristik Jaringan
Jejaring Facebook SNAP memiliki karakteristik **Small-World dengan Clustering Tinggi** (tipe *Watts-Strogatz*). Struktur ini memiliki sentralisasi tinggi pada beberapa aktor kunci seperti **Node 107**. Ini membuat jaringan sangat efisien untuk menyebarkan informasi dengan bantuan aktor kunci, namun juga rentan terfragmentasi jika aktor-aktor kunci tersebut dihapus.